# Difference in admissions for the Taunton+Worcester case

Where do patients go when the units change?

## Code setup

In [1]:
import os
import numpy as np
import pandas as pd

In [43]:
dir_data = 'data'
dir_output = 'output'
# Scenario results:
file_base_geo = 'results_geo_outcomes_base.csv'
file_base_results = 'results_full_base.csv'
file_taunworc_geo = 'results_geo_outcomes_taunworc.csv'
file_taunworc_results = 'results_full_taunworc.csv'
# Reference:
file_hospitals = 'stroke_hospitals_2025.csv'
# Output:
file_results_comparison = 'results_full_comparison_taunworc_base.csv'

## Load data

Scenario results:

In [3]:
p = os.path.join(dir_output, file_base_geo)
p2 = os.path.join(dir_output, file_base_results)
df_base = pd.concat(
    (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
    axis='columns'
)

In [4]:
p = os.path.join(dir_output, file_taunworc_geo)
p2 = os.path.join(dir_output, file_taunworc_results)
df_taunworc = pd.concat(
    (pd.read_csv(p, index_col='LSOA'), pd.read_csv(p2, index_col='LSOA')),
    axis='columns'
)

In [5]:
display(df_base.head(5))

,nearest_ivt_unit,nearest_ivt_time,nearest_ivt_unit_no_bias,nearest_ivt_time_no_bias,nearest_mt_unit,nearest_mt_time,transfer_unit,transfer_required,transfer_time,area_lower,...,drip_ship_mrs_0-2_improvement_over_no_treatment,mothership_mrs_0-2_improvement_over_no_treatment,nlvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment,lvo_drip_ship_ivt_mrs_0-2_improvement_over_no_treatment,lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment,drip_ship_utility_shift_transferloss0!6,drip_ship_mrs_0-2_transferloss0!6,drip_ship_mrs_0-2_improvement_over_no_treatment_transferloss0!6,lvo_drip_ship_ivt_mt_mrs_0-2_transferloss0!6,lvo_drip_ship_ivt_mt_mrs_0-2_improvement_over_no_treatment_transferloss0!6
LSOA,,,,,,,,,,,,,,,,,,,,,
ADUR001A,BN25BE,10.9,BN25BE,11,BN25BE,11,BN25BE,False,0,Adur 001A,...,0.134,0.133,0.121,0.06,0.164,0.12470,0.619,0.134,0.429,0.164
ADUR001B,BN25BE,11.9,BN25BE,12,BN25BE,12,BN25BE,False,0,Adur 001B,...,0.133,0.133,0.120,0.06,0.164,0.12424,0.619,0.133,0.429,0.164
ADUR001C,BN112DH,11.0,BN112DH,11,BN25BE,13,BN25BE,True,24,Adur 001C,...,0.117,0.133,0.120,0.06,0.111,0.10364,0.597,0.111,0.356,0.091
ADUR001D,BN25BE,11.9,BN25BE,12,BN25BE,12,BN25BE,False,0,Adur 001D,...,0.133,0.133,0.120,0.06,0.164,0.12424,0.619,0.133,0.429,0.164
ADUR001E,BN112DH,10.0,BN112DH,10,BN25BE,13,BN25BE,True,24,Adur 001E,...,0.118,0.133,0.121,0.06,0.112,0.10407,0.598,0.112,0.356,0.091


### Hospital reference

In [44]:
# Load hospital coordinates from CSV and create a GeoDataFrame
p = os.path.join(dir_data, file_hospitals)
df_hospitals = pd.read_csv(p, index_col='Postcode')

# Only keep hospitals in England:
df_hospitals = df_hospitals.loc[df_hospitals["Country"] == "England"]

In [45]:
display(df_hospitals.head(3))

,SSNAP name,Country,SSNAP Region,Easting,Northing,long,lat,Hospital_City,LSOA11CD,Use_IVT,Use_MT
Postcode,,,,,,,,,,,
OX39DU,John Radcliffe Hospital,England,"Buckinghamshire, Oxford and Berkshire",453940,207531,-1.219806,51.763871,Oxford,E01028534,1,1
RG15AN,Royal Berkshire Hospital,England,"Buckinghamshire, Oxford and Berkshire",472411,172962,-0.959330,51.451022,Reading,E01016414,1,0
HP112TT,Wycombe General Hospital,England,"Buckinghamshire, Oxford and Berkshire",486386,192694,-0.753428,51.626442,High Wycombe,E01017839,1,0


## Gather data

In [6]:
cols_to_gather = ['nearest_ivt_unit', 'transfer_unit', 'transfer_required']
col_admissions = 'Admissions_2025_Est'

In [8]:
rename_base = dict([(c, f'{c}_base') for c in cols_to_gather])
rename_taunworc = dict([(c, f'{c}_taunworc') for c in cols_to_gather])

df_compare = pd.concat(
    (df_base[[col_admissions]].round(1),
     df_base[cols_to_gather].rename(columns=rename_base),
     df_taunworc[cols_to_gather].rename(columns=rename_taunworc)),
    axis='columns')

In [9]:
display(df_compare.head(5))

,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,transfer_required_taunworc
LSOA,,,,,,,
ADUR001A,1.5,BN25BE,BN25BE,False,BN25BE,BN25BE,False
ADUR001B,1.9,BN25BE,BN25BE,False,BN25BE,BN25BE,False
ADUR001C,2.2,BN112DH,BN25BE,True,BN112DH,BN25BE,True
ADUR001D,2.2,BN25BE,BN25BE,False,BN25BE,BN25BE,False
ADUR001E,4.1,BN112DH,BN25BE,True,BN112DH,BN25BE,True


## Calculate admissions by unit

Calculate number of admissions in each combination of units across both scenarios:

In [10]:
cols_to_group = ['nearest_ivt_unit', 'transfer_unit']
cols_to_group = [f'{c}{s}' for s in ['_base', '_taunworc']
                 for c in cols_to_group]

In [11]:
df_admissions_unit = df_compare.groupby(cols_to_group)[col_admissions].sum()
df_admissions_unit = df_admissions_unit.reset_index()

for s in ['_base', '_taunworc']:
    df_admissions_unit[f'transfer_required{s}'] = (
        df_admissions_unit[f'transfer_unit{s}'] !=
        df_admissions_unit[f'nearest_ivt_unit{s}']
    )

## Patients affected by new MT units

Show only patients where either their nearest or transfer unit has changed:

In [12]:
m1 = (df_admissions_unit['nearest_ivt_unit_base'] !=
      df_admissions_unit['nearest_ivt_unit_taunworc'])
m2 = (df_admissions_unit['transfer_unit_base'] !=
      df_admissions_unit['transfer_unit_taunworc'])

df_admissions_unit['change'] = (m1 | m2)

In [13]:
display(df_admissions_unit[df_admissions_unit['change']])

,nearest_ivt_unit_base,transfer_unit_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,Admissions_2025_Est,transfer_required_base,transfer_required_taunworc,change
26,DT12JY,SO166YD,DT12JY,TA15DA,697.8,True,True,True
27,DT12JY,SO166YD,TA15DA,TA15DA,6.3,True,False,True
29,DY12HQ,B152TH,WR51DD,WR51DD,4.9,True,False,True
31,EX25DW,PL68DH,EX25DW,TA15DA,628.1,True,True,True
32,EX314JB,PL68DH,EX314JB,TA15DA,467.4,True,True,True
34,GL537AN,BS105NB,GL537AN,WR51DD,797.5,True,True,True
38,HR12ER,B152TH,HR12ER,WR51DD,516.8,True,True,True
90,TA15DA,BS105NB,TA15DA,TA15DA,934.9,True,False,True
98,WR51DD,B152TH,WR51DD,WR51DD,700.4,True,False,True


In [38]:
units_affected = df_admissions_unit.loc[
    df_admissions_unit['change'], 'nearest_ivt_unit_base'].unique()

# Quick check that there are no extra units in other scenario:
units_affected2 = df_admissions_unit.loc[
    df_admissions_unit['change'], 'nearest_ivt_unit_taunworc'].unique()

print((set(units_affected) - set(units_affected2)))

{'DY12HQ'}


There is one unit that is different for 4.9 patients only, so ignore it.

In [39]:
units_affected = units_affected2

In [46]:
display(df_hospitals.loc[units_affected, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
Postcode,,
DT12JY,Dorset County Hospital,Dorchester
TA15DA,Musgrove Park Hospital,Taunton
WR51DD,Worcestershire Royal Hospital,Worcester
EX25DW,Royal Devon and Exeter Hospital,Exeter
EX314JB,North Devon District Hospital,Barnstaple
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
HR12ER,Hereford County Hospital,Hereford


## Travel time bias

Some patients have an identical travel time to their two nearest units. When neither offers MT, they pick the first alphabetically by postcode. When one changes to offer MT, they pick that one.

Show affected patients:

In [14]:
m = (df_admissions_unit['nearest_ivt_unit_base'] !=
     df_admissions_unit['nearest_ivt_unit_taunworc'])

display(df_admissions_unit[df_admissions_unit['change'] & m])

,nearest_ivt_unit_base,transfer_unit_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,Admissions_2025_Est,transfer_required_base,transfer_required_taunworc,change
27,DT12JY,SO166YD,TA15DA,TA15DA,6.3,True,False,True
29,DY12HQ,B152TH,WR51DD,WR51DD,4.9,True,False,True


This is so few patients that it can only be a handful of LSOA at most.

Check numbers of LSOA:

In [15]:
display(df_compare.loc[(df_compare['nearest_ivt_unit_base'] !=
                df_compare['nearest_ivt_unit_taunworc'])])

,Admissions_2025_Est,nearest_ivt_unit_base,transfer_unit_base,transfer_required_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,transfer_required_taunworc
LSOA,,,,,,,
SOUTHSOMERSET002E,3.7,DT12JY,SO166YD,True,TA15DA,TA15DA,False
SOUTHSOMERSET017H,2.6,DT12JY,SO166YD,True,TA15DA,TA15DA,False
WYREFOREST011A,4.9,DY12HQ,B152TH,True,WR51DD,WR51DD,False


Only three LSOA are in this case.

## Transfer loss

How would this affect patients in the transfer loss case?

If patients go directly to an MT unit, the transfer rate doesn't matter. If they go to an IVT-only unit first, then they might not receive a transfer for MT.

In [16]:
mt_rate = 0.1
transfer_rate = 0.6

In [17]:
df_admissions_unit['mt'] = (
    df_admissions_unit[col_admissions] * mt_rate).round(1)
for s in ['base', 'taunworc']:
    n = f'mt_transferloss_{s}'
    df_admissions_unit[n] = df_admissions_unit['mt'].copy()
    m = f'transfer_required_{s}'
    df_admissions_unit.loc[df_admissions_unit[m], n] *= transfer_rate
    df_admissions_unit[n] = df_admissions_unit[n].round(1)

In [18]:
display(df_admissions_unit[df_admissions_unit['change']])

,nearest_ivt_unit_base,transfer_unit_base,nearest_ivt_unit_taunworc,transfer_unit_taunworc,Admissions_2025_Est,transfer_required_base,transfer_required_taunworc,change,mt,mt_transferloss_base,mt_transferloss_taunworc
26,DT12JY,SO166YD,DT12JY,TA15DA,697.8,True,True,True,69.8,41.9,41.9
27,DT12JY,SO166YD,TA15DA,TA15DA,6.3,True,False,True,0.6,0.4,0.6
29,DY12HQ,B152TH,WR51DD,WR51DD,4.9,True,False,True,0.5,0.3,0.5
31,EX25DW,PL68DH,EX25DW,TA15DA,628.1,True,True,True,62.8,37.7,37.7
32,EX314JB,PL68DH,EX314JB,TA15DA,467.4,True,True,True,46.7,28.0,28.0
34,GL537AN,BS105NB,GL537AN,WR51DD,797.5,True,True,True,79.8,47.9,47.9
38,HR12ER,B152TH,HR12ER,WR51DD,516.8,True,True,True,51.7,31.0,31.0
90,TA15DA,BS105NB,TA15DA,TA15DA,934.9,True,False,True,93.5,56.1,93.5
98,WR51DD,B152TH,WR51DD,WR51DD,700.4,True,False,True,70.0,42.0,70.0


In [20]:
cols = ['Admissions_2025_Est', 'mt', 'mt_transferloss_base',
        'mt_transferloss_taunworc']
display(df_admissions_unit.loc[df_admissions_unit['change'], cols].sum())

Admissions_2025_Est         4754.1
mt                           475.4
mt_transferloss_base         285.3
mt_transferloss_taunworc     351.1
dtype: float64

## Proportion going directly to MT-capable unit

In [34]:
def calculate_props(df):
    r = pd.Series()
    r['all'] = df[col_admissions].sum()
    for s in ['_base', '_taunworc']:
        r[f'direct{s}'] = df.loc[~df[f'transfer_required{s}'], col_admissions].sum()
        r[f'transfer{s}'] = df.loc[df[f'transfer_required{s}'], col_admissions].sum()
        r[f'prop_direct{s}'] = (r[f'direct{s}'] / r['all']).round(3)
        r[f'prop_transfer{s}'] = (r[f'transfer{s}'] / r['all']).round(3)
    return r

Whole of England:

In [35]:
calculate_props(df_admissions_unit)

all                       90659.000
direct_base               29991.500
transfer_base             60667.500
prop_direct_base              0.331
prop_transfer_base            0.669
direct_taunworc           31638.000
transfer_taunworc         59021.000
prop_direct_taunworc          0.349
prop_transfer_taunworc        0.651
dtype: float64

Anyone who goes to an affected unit as their first unit in the base case:

In [46]:
display(df_hospitals.loc[units_affected, ['SSNAP name', 'Hospital_City']])

,SSNAP name,Hospital_City
Postcode,,
DT12JY,Dorset County Hospital,Dorchester
TA15DA,Musgrove Park Hospital,Taunton
WR51DD,Worcestershire Royal Hospital,Worcester
EX25DW,Royal Devon and Exeter Hospital,Exeter
EX314JB,North Devon District Hospital,Barnstaple
GL537AN,Cheltenham & Gloucester Hospitals,Cheltenham
HR12ER,Hereford County Hospital,Hereford


In [40]:
m = df_admissions_unit['nearest_ivt_unit_base'].isin(units_affected)
calculate_props(df_admissions_unit.loc[m])

all                       4749.200
direct_base                  0.000
transfer_base             4749.200
prop_direct_base             0.000
prop_transfer_base           1.000
direct_taunworc           1641.600
transfer_taunworc         3107.600
prop_direct_taunworc         0.346
prop_transfer_taunworc       0.654
dtype: float64